# 03. Model Training and Optimization

В этом ноутбуке выполняется обучение и оптимизация моделей для
прогнозирования оттока клиентов.

На данном этапе:

- исследуются Logistic Regression, Decision Tree и Random Forest;
- выполняется подбор гиперпараметров с помощью GridSearchCV;
- оценивается влияние порога классификации;
- модели сравниваются по основным метрикам;
- выбирается конфигурация для дальнейшей финальной оценки.

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

## 1. Подготовка данных

In [40]:
df = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df.loc[
    df["tenure"] == 0,
    "TotalCharges"
] = 0

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

X = df.drop(
    columns=["Churn", "customerID"]
)

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



In [42]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns

C:\Users\Admin\AppData\Local\Temp\ipykernel_23668\2015223730.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [43]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

## 2. Logistic Regression — Hyperparameter Tuning

На первом этапе оптимизируем Logistic Regression.

Подбираются два гиперпараметра:

- `C` — коэффициент регуляризации;
- `class_weight` — способ учёта дисбаланса классов.

В качестве целевой метрики используется F1-score.
Для оценки используется стратифицированная 5-fold кросс-валидация.

In [44]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=10000
            )
        )
    ]
)


In [45]:
logistic_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

In [46]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [47]:
logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

In [48]:
logistic_grid.fit(
    X_train,
    y_train
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...iter=10000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__class_weight': [None, 'balanced']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, defau

In [49]:
print("Best parameters:")
print(logistic_grid.best_params_)

print("\nBest CV F1:")
print(logistic_grid.best_score_)

Best parameters:
{'model__C': 0.1, 'model__class_weight': 'balanced'}

Best CV F1:
0.6297476916621374


In [50]:
logistic_results = (
    pd.DataFrame(logistic_grid.cv_results_)
    [
        [
            "param_model__C",
            "param_model__class_weight",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ]
    .sort_values("rank_test_score")
)

logistic_results

,param_model__C,param_model__class_weight,mean_test_score,std_test_score,rank_test_score
3,0.10,balanced,0.629748,0.021996,1
5,1.00,balanced,0.629617,0.021795,2
1,0.01,balanced,0.628956,0.020228,3
7,10.00,balanced,0.627896,0.023928,4
9,100.00,balanced,0.626708,0.023839,5
6,10.00,NaN,0.597540,0.030248,6
8,100.00,NaN,0.597378,0.028765,7
4,1.00,NaN,0.592563,0.030315,8
2,0.10,NaN,0.589966,0.028007,9
0,0.01,NaN,0.564923,0.018741,10


In [51]:
best_logistic = logistic_grid.best_estimator_

In [52]:
y_proba_logistic = best_logistic.predict_proba(
    X_test
)[:, 1]

In [53]:
y_pred_logistic = (
    y_proba_logistic >= 0.5
).astype(int)

In [54]:
logistic_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_logistic),
    "Precision": precision_score(y_test, y_pred_logistic),
    "Recall": recall_score(y_test, y_pred_logistic),
    "F1": f1_score(y_test, y_pred_logistic),
    "ROC-AUC": roc_auc_score(y_test, y_proba_logistic)
}

logistic_metrics

{'Accuracy': 0.7423704755145494,
 'Precision': 0.5095652173913043,
 'Recall': 0.7834224598930482,
 'F1': 0.6174920969441517,
 'ROC-AUC': 0.8413908910072593}

## 3. Decision Tree — Hyperparameter Tuning

На втором этапе исследуется Decision Tree.

Подбираются:

- `max_depth` — максимальная глубина дерева;
- `min_samples_leaf` — минимальное количество объектов в листе.

В качестве целевой метрики используется F1-score.
Для подбора применяется стратифицированная 5-fold cross-validation.

In [55]:
tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

In [56]:
tree_param_grid = {
    "model__max_depth": [2, 3, 4, 5, 6, 8, 10],
    "model__min_samples_leaf": [1, 2, 5, 10, 20]
}

In [57]:
tree_grid = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=tree_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)


In [58]:
tree_grid.fit(
    X_train,
    y_train
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, defau

In [59]:
print("Best parameters:")
print(tree_grid.best_params_)

print("\nBest CV F1:")
print(tree_grid.best_score_)

Best parameters:
{'model__max_depth': 5, 'model__min_samples_leaf': 5}

Best CV F1:
0.5772959167517328


In [60]:
best_tree = tree_grid.best_estimator_

In [61]:
tree_results = (
    pd.DataFrame(tree_grid.cv_results_)
    [
        [
            "param_model__max_depth",
            "param_model__min_samples_leaf",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ]
    .sort_values("rank_test_score")
)

tree_results.head(10)

,param_model__max_depth,param_model__min_samples_leaf,mean_test_score,std_test_score,rank_test_score
17,5,5,0.577296,0.033255,1
18,5,10,0.577212,0.033216,2
19,5,20,0.577212,0.033216,2
15,5,1,0.576555,0.032472,4
16,5,2,0.576555,0.032472,4
28,8,10,0.554832,0.024472,6
34,10,20,0.551734,0.029722,7
29,8,20,0.550388,0.017509,8
26,8,2,0.541479,0.025195,9
21,6,2,0.541234,0.032424,10


In [62]:
y_proba_tree = best_tree.predict_proba(
    X_test
)[:, 1]

y_pred_tree = (
    y_proba_tree >= 0.5
).astype(int)

In [63]:
tree_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_tree),
    "Precision": precision_score(y_test, y_pred_tree),
    "Recall": recall_score(y_test, y_pred_tree),
    "F1": f1_score(y_test, y_pred_tree),
    "ROC-AUC": roc_auc_score(y_test, y_proba_tree)
}

tree_metrics

{'Accuracy': 0.7955997161107168,
 'Precision': 0.6335403726708074,
 'Recall': 0.5454545454545454,
 'F1': 0.5862068965517241,
 'ROC-AUC': 0.8271874757808262}

## 4. Random Forest — Hyperparameter Tuning

На третьем этапе исследуется Random Forest.

Подбираются следующие гиперпараметры:

- `n_estimators` — количество деревьев;
- `max_depth` — максимальная глубина деревьев;
- `min_samples_leaf` — минимальное количество объектов в листе.

В качестве целевой метрики используется F1-score.
Для подбора применяется стратифицированная 5-fold cross-validation.

In [64]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [65]:
rf_param_grid = {
    "model__n_estimators": [100, 200, 500],
    "model__max_depth": [6, 8, 10],
    "model__min_samples_leaf": [1, 5, 10]
}

In [66]:
rf_grid = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=rf_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

In [67]:
rf_grid.fit(
    X_train,
    y_train
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [6, 8, ...], 'model__min_samples_leaf': [1, 5, ...], 'model__n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable ad

In [68]:
print("Best parameters:")
print(rf_grid.best_params_)

print("\nBest CV F1:")
print(rf_grid.best_score_)

Best parameters:
{'model__max_depth': 10, 'model__min_samples_leaf': 5, 'model__n_estimators': 500}

Best CV F1:
0.5714258215115497


In [69]:
rf_results = (
    pd.DataFrame(rf_grid.cv_results_)
    [
        [
            "param_model__n_estimators",
            "param_model__max_depth",
            "param_model__min_samples_leaf",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ]
    .sort_values("rank_test_score")
)

rf_results.head(10)

,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,mean_test_score,std_test_score,rank_test_score
23,500,10,5,0.571426,0.019606,1
26,500,10,10,0.569412,0.016607,2
22,200,10,5,0.567959,0.019053,3
20,500,10,1,0.567428,0.022883,4
18,100,10,1,0.567077,0.020803,5
12,100,8,5,0.565498,0.022700,6
25,200,10,10,0.564916,0.016698,7
13,200,8,5,0.564516,0.018796,8
15,100,8,10,0.564360,0.018850,9
16,200,8,10,0.563501,0.020231,10


In [70]:
best_rf = rf_grid.best_estimator_

In [71]:
y_proba_rf = best_rf.predict_proba(
    X_test
)[:, 1]

y_pred_rf = (
    y_proba_rf >= 0.5
).astype(int)

In [72]:
rf_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Precision": precision_score(y_test, y_pred_rf),
    "Recall": recall_score(y_test, y_pred_rf),
    "F1": f1_score(y_test, y_pred_rf),
    "ROC-AUC": roc_auc_score(y_test, y_proba_rf)
}

rf_metrics

{'Accuracy': 0.8041163946061036,
 'Precision': 0.6701388888888888,
 'Recall': 0.516042780748663,
 'F1': 0.5830815709969789,
 'ROC-AUC': 0.8441757730760289}

In [73]:
model_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        **logistic_metrics
    },
    {
        "Model": "Decision Tree",
        **tree_metrics
    },
    {
        "Model": "Random Forest",
        **rf_metrics
    }
])

model_comparison

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.742370,0.509565,0.783422,0.617492,0.841391
1,Decision Tree,0.795600,0.633540,0.545455,0.586207,0.827187
2,Random Forest,0.804116,0.670139,0.516043,0.583082,0.844176


## 5. Out-of-Fold Predictions и подбор threshold

После подбора гиперпараметров исследуем влияние порога классификации.

Для выбора threshold используются out-of-fold (OOF) предсказания на
обучающей выборке.

При OOF-предсказаниях каждый объект получает вероятность от модели,
которая не использовала этот объект при обучении.

Это позволяет подобрать threshold без использования тестовой выборки.

In [74]:
oof_proba_logistic = cross_val_predict(
    best_logistic,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

In [75]:
oof_proba_tree = cross_val_predict(
    best_tree,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

In [76]:
oof_proba_rf = cross_val_predict(
    best_rf,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

In [80]:
def evaluate_thresholds(y_true, y_proba):

    thresholds = np.arange(0.2,0.9, 0.1)

    results = []

    for threshold in thresholds:
        y_pred = (
            y_proba >= threshold
        ).astype(int)

        results.append({
            "Threshold" : round(threshold, 1),
            "Accuracy" : accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "F1 score": f1_score(y_true, y_pred)
        })

    return pd.DataFrame(results)

In [81]:
logistic_thresholds = evaluate_thresholds(
    y_train,
    oof_proba_logistic
)

tree_thresholds = evaluate_thresholds(
    y_train,
    oof_proba_tree
)

rf_thresholds = evaluate_thresholds(
    y_train,
    oof_proba_rf
)

In [82]:
logistic_thresholds

,Threshold,Accuracy,Precision,Recall,F1 score
0,0.2,0.574015,0.380323,0.961873,0.545110
1,0.3,0.645722,0.423465,0.927090,0.581376
2,0.4,0.708555,0.473047,0.862876,0.611085
3,0.5,0.751509,0.520770,0.796656,0.629825
4,0.6,0.783103,0.573904,0.709030,0.634351
5,0.7,0.799610,0.635155,0.575251,0.603721
6,0.8,0.796770,0.717122,0.386622,0.502390


In [84]:
tree_thresholds

,Threshold,Accuracy,Precision,Recall,F1 score
0,0.2,0.711040,0.474877,0.840803,0.606953
1,0.3,0.751331,0.521739,0.754515,0.616899
2,0.4,0.782570,0.583747,0.629431,0.605729
3,0.5,0.789847,0.618251,0.543813,0.578648
4,0.6,0.785410,0.724138,0.309030,0.433193
5,0.7,0.779730,0.745174,0.258194,0.383507
6,0.8,0.751864,0.821192,0.082943,0.150668


In [85]:
rf_thresholds

,Threshold,Accuracy,Precision,Recall,F1 score
0,0.2,0.700568,0.465861,0.876254,0.608312
1,0.3,0.765353,0.540821,0.766555,0.634200
2,0.4,0.796592,0.612508,0.635452,0.623769
3,0.5,0.803159,0.676740,0.494314,0.571318
4,0.6,0.788782,0.723280,0.330435,0.453627
5,0.7,0.775825,0.816940,0.200000,0.321333
6,0.8,0.751686,0.880952,0.074247,0.136952


In [90]:
def get_best_threshold(results):

    best_threshold = results.loc[
        results["F1 score"].idxmax(),
        "Threshold"
    ]
    return best_threshold

In [91]:
best_threshold_logistic = get_best_threshold(
    logistic_thresholds
)

best_threshold_tree = get_best_threshold(
    tree_thresholds
)

best_threshold_rf = get_best_threshold(
    rf_thresholds
)

In [92]:
print("Logistic Regression:")
print(best_threshold_logistic)

print("\nDecision Tree:")
print(best_threshold_tree)

print("\nRandom Forest:")
print(best_threshold_rf)

Logistic Regression:
0.6

Decision Tree:
0.3

Random Forest:
0.3


## 6. Сравнение моделей

После подбора гиперпараметров и threshold сравним итоговые конфигурации
трёх моделей на тестовой выборке.

Для каждой модели используется threshold, выбранный на OOF-предсказаниях.

In [98]:
fitted_models = {
    "Logistic Regression": best_logistic,
    "Decision Tree": best_tree,
    "Random Forest": best_rf
}

final_thresholds = {
    "Logistic Regression": best_threshold_logistic,
    "Decision Tree": best_threshold_tree,
    "Random Forest": best_threshold_rf
}

In [99]:
final_results = []

for name, model in fitted_models.items():

    threshold = final_thresholds[name]

    y_proba = model.predict_proba(X_test)[:,1]
    
    y_pred = (
        y_proba >= threshold
    ).astype(int)

    final_results.append({
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })


In [100]:
final_results = pd.DataFrame(final_results)
final_results

,Model,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.6,0.763662,0.542620,0.697861,0.610526,0.841391
1,Decision Tree,0.3,0.732434,0.497400,0.767380,0.603575,0.827187
2,Random Forest,0.3,0.757275,0.529197,0.775401,0.629067,0.844176


## 7. Final Model Selection

По результатам подбора гиперпараметров, настройки порога классификации
и сравнения моделей на тестовой выборке были получены следующие результаты:

- Logistic Regression: F1 = 0.611, ROC-AUC = 0.841;
- Decision Tree: F1 = 0.604, ROC-AUC = 0.827;
- Random Forest: F1 = 0.629, ROC-AUC = 0.844.

Random Forest показывает наилучший результат по F1-score и ROC-AUC,
а также обеспечивает высокий Recall.

В качестве финальной модели выбирается Random Forest со следующими
параметрами:

- `n_estimators = 500`;
- `max_depth = 10`;
- `min_samples_leaf = 10`;
- `threshold = 0.3`.

Финальная модель будет дополнительно исследована в следующем ноутбуке.